## **Langchain 지원 청킹(Chunking) 클래스와 메소드**
- CharacterTextSplitter
    - 텍스트를 지정된 기준으로 나눠주는 기본적인 클래스
    - 간단한 구조일 때 적절 (잘 정돈된 텍스트, csv 파일)
    - 구분자는 디폴트로 '\n\n'을 사용하며 변경 가능
    - 구분자가 단일 문자이기 때문에 연속된 텍스트의 의미를 충분히 고려하지 못할 수도 있음
- RecursiveCharacterTextSplitter
    - 텍스트를 여러 단계를 거쳐 `계층적으로` 분리해주는 클래스 (청킹에서 가장 많이 활용됨)
    - 상대적으로 복잡한 구조일 때 적절 (일반적인 문서, 웹페이지, 보고서)
    - 리스트를 통해 구분자를 여러개 사용하여 세분화된 청크로 만들 수 있음
    - 처음엔 '\n\n'으로 분리하고 청크가 여전히 크면 '\n' 그래도 크면 '' 이런 식으로 계층적 분할 진행

**간단한 텍스트 처리에는 CharacterTextSplitter가 Recursive에 비해 속도가 빨라 대용량 처리에 유리함**

> 그 외에도 토큰 단위로 잘라주는 ToeknTextSplitter, 임베딩 기술 써서 비슷한 문장끼리 자르는 SemanticChunker 있음

In [8]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import re


In [9]:
text1 = '청킹을 위한 예제 데이터입니다. 어떻게 쪼개는지 알아볼까요?'
text2 = '청킹을 위한 예제 데이터입니다. 어떻게 쪼개는지 알아볼까요? 결과를 확인해봅시다.'
text3 = '''청킹을 위한 예제 데이터입니다. 
어떻게 쪼개는지 알아볼까요? 

결과를 확인해봅시다.'''

### 1) Character Splitter

In [11]:
chunk_size=20    # 한 청크에 대한 길이 기준을 문자 20으로 지정
chunk_overlap=0

# 청킹 객체 생성
c_splitter = CharacterTextSplitter(
    # 청크 크기 지정
     # (한 청크당 의미있는 문맥을 유지하기 위해서 일반 문서에서는 200~1500 사이로 지정)
     # 데이터의 종류에 따라 달라짐 (기술문서/연구논문은 상대적으로 크게, 뉴스기사나 소설은 상대적으로 작게)
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)


In [15]:
# split_text : 텍스트 데이터 청킹 실행 메소드
c_splitter.split_text(text1)

['청킹을 위한 예제 데이터입니다. 어떻게 쪼개는지 알아볼까요?']

In [16]:
c_splitter.split_text(text2)

['청킹을 위한 예제 데이터입니다. 어떻게 쪼개는지 알아볼까요? 결과를 확인해봅시다.']

In [ ]:
c_splitter.split_text(text3)

# 무조건 설정한 chunk_size보다 작은 크기로 분리하는 건 아니고,
# 의미있는 문장이라 판단되면 splitter의 조건에 따라 더 긴 청크 반환

Created a chunk of size 35, which is longer than the specified 20


['청킹을 위한 예제 데이터입니다. \n어떻게 쪼개는지 알아볼까요?', '결과를 확인해봅시다.']

### 2) Recursive Splitter

In [25]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

In [21]:
r_splitter.split_text(text1)

['청킹을 위한 예제 데이터입니다.', '어떻게 쪼개는지 알아볼까요?']

In [26]:
r_splitter.split_text(text2)

['청킹을 위한 예제 데이터입니다.', '어떻게 쪼개는지 알아볼까요? 결과를', '확인해봅시다.']

In [23]:
r_splitter.split_text(text3)

['청킹을 위한 예제 데이터입니다.', '어떻게 쪼개는지 알아볼까요?', '결과를 확인해봅시다.']

### 3) Character, Recursive 비교

In [27]:
see_you_again = """It's been a long day without you, my friend.
And I'll tell you all about it when I see you again.
We've come a long way from where we began.
Oh, I'll tell you all about it when I see you again.
When I see you again.


Damn, who knew?
All the planes we flew, good things we been through.
That I'd be standing right here talking to you.
'Bout another path, I know we loved to hit the road and laugh.
But something told me that it wouldn't last.
Had to switch up, look at things different, see the bigger picture.
Those were the days, hard work forever pays.
Now I see you in a better place (see you in a better place).
Uh."""

In [30]:
c_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separator='\n\n'  # 변경 가능
)
c_splitter.split_text(see_you_again)

Created a chunk of size 215, which is longer than the specified 100


["It's been a long day without you, my friend.\nAnd I'll tell you all about it when I see you again.\nWe've come a long way from where we began.\nOh, I'll tell you all about it when I see you again.\nWhen I see you again.",
 "Damn, who knew?\nAll the planes we flew, good things we been through.\nThat I'd be standing right here talking to you.\n'Bout another path, I know we loved to hit the road and laugh.\nBut something told me that it wouldn't last.\nHad to switch up, look at things different, see the bigger picture.\nThose were the days, hard work forever pays.\nNow I see you in a better place (see you in a better place).\nUh."]

In [32]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    # 리스트로 계층적 조건을 설정 (더 큰 범위를 먼저 지정해야 함)
    separators=['\n\n', '\n', ' ']
)
r_splitter.split_text(see_you_again)

["It's been a long day without you, my friend.\nAnd I'll tell you all about it when I see you again.",
 "We've come a long way from where we began.\nOh, I'll tell you all about it when I see you again.",
 'When I see you again.',
 'Damn, who knew?\nAll the planes we flew, good things we been through.',
 "That I'd be standing right here talking to you.",
 "'Bout another path, I know we loved to hit the road and laugh.",
 "But something told me that it wouldn't last.",
 'Had to switch up, look at things different, see the bigger picture.',
 'Those were the days, hard work forever pays.',
 'Now I see you in a better place (see you in a better place).\nUh.']

## **문서 로더(Document Loader)**
- langchain의 문서 로더 기능을 사용해 다양한 형식의 데이터 파일로부터 텍스트 로드
- 로드한 문서는 `Document 객체로 반환`되며, page_content에는 텍스트 내용, metadata에는 데이터에 대한 설명을 포함함
- pdf 파일 외에도 Word, CSV, Excel, JSON 등을 위한 문서 로더가 각각 존재
- <mark>LLM은 파일 자체를 통째로 읽지 못함. 그래서 파일 열고(load), 글자 뽑아서(extraction), Document 객체로 만들어줘야함</mark>

#### 종류
- WebBaseLoader: 웹페이지 문서 로드
- TextLoader : TXT 파일
- DirectoryLoader : 특정 경로에 있는 여러개의 파일 로드
- PyPDFLoader
- JSONLoader
- CSVLoader

### 1) WebBaseLoader

In [ ]:
# WebBaseLoader : 랭체인 지원 웹페이지 HTML 컨텐츠를 로드해주는 클래스
#                 내부적으로 BeautifulSoup 사용함
from langchain_community.document_loaders import WebBaseLoader
import bs4
# warning은 웹 접근시 user_agent 설정 필요하다는 문구

In [ ]:
url = 'https://www.aitimes.com/news/articleView.html?idxno=208020'

loader = WebBaseLoader(
    # 웹페이지 url
    web_path=[url],
    # bs_kwargs : bs 세부사항 설정 (옵션 보따리~!🎁)
     # parse_only : 파싱 설정
      # SoupStrainer : HTML 문서에서 특정 조건에 맞는 요소만 선택적으로 파싱하는 함수
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            # 'article'            ## article 태그만 가져오겠다
            attrs={'class':['article-body']} 

            # 특정 클래스나 ID를 가진 태그를 가져올 경우
            # 'div', attrs={'class':['클래스명1', '클래스명2', ...]}
        )
    )
)

# 브라우저 인증
loader.requests_kwargs={'headers':{'User-Agent':'Mozilla/5.0'}}

# 한글 인코딩 설정
loader.encoding='utf-8'

docs = loader.load()
docs

[Document(metadata={'source': 'https://www.aitimes.com/news/articleView.html?idxno=208020'}, page_content="\n\n\n\n\xa0\xa0기사를 읽어드립니다.\r\n\t\t\t\t\n\n\n실존 도시를 기반으로 한 AI 시뮬레이션이 현실화되고 있다. 가상의 장면을 만들어내던 기존 생성\xa0모델과 달리, 실제 도시를 그대로 재현하고 자유롭게 탐색할 수 있는 새로운 형태의 ‘월드 모델’이 등장했다.\n네이버와 한국과학기술원(KAIST), 서울대학교 연구진은 16일 서울을 기반으로 한 도시 규모의 생성 모델 ‘서울 월드 모델(SWM·Seoul World Mode)’을 온라인 아카이브를 통해 공개했다.\n이 모델은 단순히 그럴듯한 장면을 만들어내는 것이 아니라, 실제 존재하는 도시의 거리와 건물 배치를 반영해 영상을 생성하는 것이 핵심 특징이다.\xa0\n기존 영상 생성 모델들은 보이지 않는 영역까지 모두 상상해 만들어내는 방식이었다. 하지만,\xa0SWM은 실제 서울의 거리 데이터를 기반으로 작동한다. 특정 좌표와 카메라 움직임, 텍스트 명령을 입력하면, 해당 위치 주변의 스트리트뷰 이미지를 불러와 이를 기반으로 영상을 생성한다.\n이 과정에서 모델은 단순히 이미지를 이어붙이는 것이 아니라, 공간 구조와 시각적 특징을 동시에 반영해 새로운 영상을 만들어낸다. 이를 통해 실제 도시와 일치하는 공간적 정확성과 시간적 일관성을 확보했다.\n\n\n\nSWM 개요. 시작 위치가 주어지면, SWM은 텍스트 프롬프트 P(i)와 목표 카메라 궤적 C(i)을 기반으로 실제 도시에 정합된 영상을 자기회귀적으로 생성하며, 지리 인덱스 기반 데이터베이스에서 관련 스트리트뷰 이미지를 검색한다. (사진=arXiv)\n\n\n기존 모델은 시간이 길어질수록 품질이 급격히 떨어지는 문제가 있었지만, SWM은 ‘가상 예측 싱크(Virtual Lookahead Sink)’라는 구조를 통해 이를 해결했다. 이 방식은

In [50]:
docs[0].metadata

{'source': 'https://www.aitimes.com/news/articleView.html?idxno=208020'}

In [55]:
print(docs[0].page_content)





  기사를 읽어드립니다.
				


실존 도시를 기반으로 한 AI 시뮬레이션이 현실화되고 있다. 가상의 장면을 만들어내던 기존 생성 모델과 달리, 실제 도시를 그대로 재현하고 자유롭게 탐색할 수 있는 새로운 형태의 ‘월드 모델’이 등장했다.
네이버와 한국과학기술원(KAIST), 서울대학교 연구진은 16일 서울을 기반으로 한 도시 규모의 생성 모델 ‘서울 월드 모델(SWM·Seoul World Mode)’을 온라인 아카이브를 통해 공개했다.
이 모델은 단순히 그럴듯한 장면을 만들어내는 것이 아니라, 실제 존재하는 도시의 거리와 건물 배치를 반영해 영상을 생성하는 것이 핵심 특징이다. 
기존 영상 생성 모델들은 보이지 않는 영역까지 모두 상상해 만들어내는 방식이었다. 하지만, SWM은 실제 서울의 거리 데이터를 기반으로 작동한다. 특정 좌표와 카메라 움직임, 텍스트 명령을 입력하면, 해당 위치 주변의 스트리트뷰 이미지를 불러와 이를 기반으로 영상을 생성한다.
이 과정에서 모델은 단순히 이미지를 이어붙이는 것이 아니라, 공간 구조와 시각적 특징을 동시에 반영해 새로운 영상을 만들어낸다. 이를 통해 실제 도시와 일치하는 공간적 정확성과 시간적 일관성을 확보했다.



SWM 개요. 시작 위치가 주어지면, SWM은 텍스트 프롬프트 P(i)와 목표 카메라 궤적 C(i)을 기반으로 실제 도시에 정합된 영상을 자기회귀적으로 생성하며, 지리 인덱스 기반 데이터베이스에서 관련 스트리트뷰 이미지를 검색한다. (사진=arXiv)


기존 모델은 시간이 길어질수록 품질이 급격히 떨어지는 문제가 있었지만, SWM은 ‘가상 예측 싱크(Virtual Lookahead Sink)’라는 구조를 통해 이를 해결했다. 이 방식은 미래 위치의 이미지를 지속적으로 참조해 영상 생성 과정에서 기준점을 유지하도록 하는 기술로, 장거리에서도 공간 일관성을 확보한다.
또 다른 강점은 다양한 카메라 이동을 지원한다는 점이다. 단순히 자동차 주행 시점에 국한되지 않고, 보행자 시점, 교차로 이동, 자

In [56]:
# 문자열 함수 replace를 활용한 텍스트 정제
docs[0].page_content.replace('\n', '')

"\xa0\xa0기사를 읽어드립니다.\r\t\t\t\t실존 도시를 기반으로 한 AI 시뮬레이션이 현실화되고 있다. 가상의 장면을 만들어내던 기존 생성\xa0모델과 달리, 실제 도시를 그대로 재현하고 자유롭게 탐색할 수 있는 새로운 형태의 ‘월드 모델’이 등장했다.네이버와 한국과학기술원(KAIST), 서울대학교 연구진은 16일 서울을 기반으로 한 도시 규모의 생성 모델 ‘서울 월드 모델(SWM·Seoul World Mode)’을 온라인 아카이브를 통해 공개했다.이 모델은 단순히 그럴듯한 장면을 만들어내는 것이 아니라, 실제 존재하는 도시의 거리와 건물 배치를 반영해 영상을 생성하는 것이 핵심 특징이다.\xa0기존 영상 생성 모델들은 보이지 않는 영역까지 모두 상상해 만들어내는 방식이었다. 하지만,\xa0SWM은 실제 서울의 거리 데이터를 기반으로 작동한다. 특정 좌표와 카메라 움직임, 텍스트 명령을 입력하면, 해당 위치 주변의 스트리트뷰 이미지를 불러와 이를 기반으로 영상을 생성한다.이 과정에서 모델은 단순히 이미지를 이어붙이는 것이 아니라, 공간 구조와 시각적 특징을 동시에 반영해 새로운 영상을 만들어낸다. 이를 통해 실제 도시와 일치하는 공간적 정확성과 시간적 일관성을 확보했다.SWM 개요. 시작 위치가 주어지면, SWM은 텍스트 프롬프트 P(i)와 목표 카메라 궤적 C(i)을 기반으로 실제 도시에 정합된 영상을 자기회귀적으로 생성하며, 지리 인덱스 기반 데이터베이스에서 관련 스트리트뷰 이미지를 검색한다. (사진=arXiv)기존 모델은 시간이 길어질수록 품질이 급격히 떨어지는 문제가 있었지만, SWM은 ‘가상 예측 싱크(Virtual Lookahead Sink)’라는 구조를 통해 이를 해결했다. 이 방식은 미래 위치의 이미지를 지속적으로 참조해 영상 생성 과정에서 기준점을 유지하도록 하는 기술로, 장거리에서도 공간 일관성을 확보한다.또 다른 강점은 다양한 카메라 이동을 지원한다는 점이다. 단순히 자동차 주행 시점에 국한되지 않고, 보행자 시점, 교차로 이동,

In [ ]:
# 문자열 정규표현식을 활용한 텍스트 정제
 # [\w]+ word가 1개 이상 반복되면 (단어 찾겠단 의미)
 # ' '.join 으로 나눠진 단어들 합쳐줘
' '.join(re.findall(r'[\w]+', docs[0].page_content))

'기사를 읽어드립니다 실존 도시를 기반으로 한 AI 시뮬레이션이 현실화되고 있다 가상의 장면을 만들어내던 기존 생성 모델과 달리 실제 도시를 그대로 재현하고 자유롭게 탐색할 수 있는 새로운 형태의 월드 모델 이 등장했다 네이버와 한국과학기술원 KAIST 서울대학교 연구진은 16일 서울을 기반으로 한 도시 규모의 생성 모델 서울 월드 모델 SWM Seoul World Mode 을 온라인 아카이브를 통해 공개했다 이 모델은 단순히 그럴듯한 장면을 만들어내는 것이 아니라 실제 존재하는 도시의 거리와 건물 배치를 반영해 영상을 생성하는 것이 핵심 특징이다 기존 영상 생성 모델들은 보이지 않는 영역까지 모두 상상해 만들어내는 방식이었다 하지만 SWM은 실제 서울의 거리 데이터를 기반으로 작동한다 특정 좌표와 카메라 움직임 텍스트 명령을 입력하면 해당 위치 주변의 스트리트뷰 이미지를 불러와 이를 기반으로 영상을 생성한다 이 과정에서 모델은 단순히 이미지를 이어붙이는 것이 아니라 공간 구조와 시각적 특징을 동시에 반영해 새로운 영상을 만들어낸다 이를 통해 실제 도시와 일치하는 공간적 정확성과 시간적 일관성을 확보했다 SWM 개요 시작 위치가 주어지면 SWM은 텍스트 프롬프트 P i 와 목표 카메라 궤적 C i 을 기반으로 실제 도시에 정합된 영상을 자기회귀적으로 생성하며 지리 인덱스 기반 데이터베이스에서 관련 스트리트뷰 이미지를 검색한다 사진 arXiv 기존 모델은 시간이 길어질수록 품질이 급격히 떨어지는 문제가 있었지만 SWM은 가상 예측 싱크 Virtual Lookahead Sink 라는 구조를 통해 이를 해결했다 이 방식은 미래 위치의 이미지를 지속적으로 참조해 영상 생성 과정에서 기준점을 유지하도록 하는 기술로 장거리에서도 공간 일관성을 확보한다 또 다른 강점은 다양한 카메라 이동을 지원한다는 점이다 단순히 자동차 주행 시점에 국한되지 않고 보행자 시점 교차로 이동 자유로운 카메라 경로 등 다양한 시나리오를 구현할 수 있다 이는 실제 데이터의 한계를 보완하기 위해 

### 2) TextLoader

In [62]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('data\인공지능이 그리는 새로운 인류의 지평.txt')

doc = loader.load()
doc

[Document(metadata={'source': 'data\\인공지능이 그리는 새로운 인류의 지평.txt'}, page_content=" 인류 역사는 도구의 발명과 함께 발전해 왔다. 불의 발견부터 증기기관, 그리고 인터넷에 이르기까지 새로운 기술은 언제나 인류의 삶을 근본적으로 바꾸어 놓았다. 그러나 지금 우리가 마주하고 있는 인공지능(AI)이라는 파도는 과거의 그 어떤 변화와도 궤를 달리한다. 과거의 기술이 인간의 '근력'이나 '속도'를 보완했다면, 인공지능은 인간 고유의 영역이라 믿어왔던 '지능'과 '창의성'에 도전하고 있기 때문이다.\n\n 기술의 진화와 패러다임의 변화 21세기 초반, 딥러닝 기술의 비약적인 발전은 인공지능을 단순한 알고리즘의 집합에서 '스스로 학습하는 존재'로 탈바꿈시켰다. 초기 인공지능이 바둑이나 체스 같은 정해진 규칙 안에서 승리하는 법을 배웠다면, 현재의 생성형 AI는 수조 개의 데이터를 학습하여 인간과 유사한 문장을 구사하고, 예술 작품을 창조하며, 복잡한 코드를 설계한다. 이러한 변화는 단순히 '똑똑한 도구'의 등장을 넘어, 정보의 생산과 소비 방식 자체를 뒤흔드는 패러다임의 전환을 의미한다. 이제 지식은 소유하는 것이 아니라, AI라는 인터페이스를 통해 어떻게 끌어내느냐(Prompt Engineering)의 문제가 되었다.\n\n 산업 구조의 재편과 새로운 기회 경제적 관점에서 인공지능은 생산성을 극대화하는 기폭제다. 제조, 물류, 금융, 의료 등 모든 산업 분야에서 AI는 효율성을 높이고 비용을 절감한다. 특히 의료 분야에서 AI는 방대한 임상 데이터를 분석해 인간 의사가 발견하지 못한 질병의 전조를 찾아내고, 신약 개발 기간을 획기적으로 단축하고 있다. 물론 이러한 효율성이 일자리 감소에 대한 공포를 불러일으키는 것도 사실이다. 단순 반복적인 업무는 AI와 로봇이 대체할 가능성이 크기 때문이다. 그러나 역사적으로 기술 혁신은 언제나 기존의 직업군을 소멸시키는 동시에, 데이터 과학자나 AI 윤리 전문가와 같은 새로운

In [63]:
doc[0].page_content

" 인류 역사는 도구의 발명과 함께 발전해 왔다. 불의 발견부터 증기기관, 그리고 인터넷에 이르기까지 새로운 기술은 언제나 인류의 삶을 근본적으로 바꾸어 놓았다. 그러나 지금 우리가 마주하고 있는 인공지능(AI)이라는 파도는 과거의 그 어떤 변화와도 궤를 달리한다. 과거의 기술이 인간의 '근력'이나 '속도'를 보완했다면, 인공지능은 인간 고유의 영역이라 믿어왔던 '지능'과 '창의성'에 도전하고 있기 때문이다.\n\n 기술의 진화와 패러다임의 변화 21세기 초반, 딥러닝 기술의 비약적인 발전은 인공지능을 단순한 알고리즘의 집합에서 '스스로 학습하는 존재'로 탈바꿈시켰다. 초기 인공지능이 바둑이나 체스 같은 정해진 규칙 안에서 승리하는 법을 배웠다면, 현재의 생성형 AI는 수조 개의 데이터를 학습하여 인간과 유사한 문장을 구사하고, 예술 작품을 창조하며, 복잡한 코드를 설계한다. 이러한 변화는 단순히 '똑똑한 도구'의 등장을 넘어, 정보의 생산과 소비 방식 자체를 뒤흔드는 패러다임의 전환을 의미한다. 이제 지식은 소유하는 것이 아니라, AI라는 인터페이스를 통해 어떻게 끌어내느냐(Prompt Engineering)의 문제가 되었다.\n\n 산업 구조의 재편과 새로운 기회 경제적 관점에서 인공지능은 생산성을 극대화하는 기폭제다. 제조, 물류, 금융, 의료 등 모든 산업 분야에서 AI는 효율성을 높이고 비용을 절감한다. 특히 의료 분야에서 AI는 방대한 임상 데이터를 분석해 인간 의사가 발견하지 못한 질병의 전조를 찾아내고, 신약 개발 기간을 획기적으로 단축하고 있다. 물론 이러한 효율성이 일자리 감소에 대한 공포를 불러일으키는 것도 사실이다. 단순 반복적인 업무는 AI와 로봇이 대체할 가능성이 크기 때문이다. 그러나 역사적으로 기술 혁신은 언제나 기존의 직업군을 소멸시키는 동시에, 데이터 과학자나 AI 윤리 전문가와 같은 새로운 직업군을 창출해 왔다. 중요한 것은 노동의 '종말'이 아니라 노동의 '성격'이 변화하고 있다는 점이다.\n\n 윤리적 과제와 책임 있는 발

In [ ]:
# splitter 적용
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# load_and_split : 데이터 로드 + 설정된 splitter 적용
docs_split = loader.load_and_split(r_splitter)
print(f'청크 개수: {len(docs_split)}')
print()

for doc in docs_split :
    print(doc.page_content)
    print('='*100)

청크 개수: 9

인류 역사는 도구의 발명과 함께 발전해 왔다. 불의 발견부터 증기기관, 그리고 인터넷에 이르기까지 새로운 기술은 언제나 인류의 삶을 근본적으로 바꾸어 놓았다. 그러나 지금 우리가 마주하고 있는 인공지능(AI)이라는 파도는 과거의 그 어떤 변화와도 궤를 달리한다. 과거의 기술이 인간의 '근력'이나 '속도'를 보완했다면, 인공지능은 인간 고유의 영역이라 믿어왔던 '지능'과 '창의성'에 도전하고 있기 때문이다.
기술의 진화와 패러다임의 변화 21세기 초반, 딥러닝 기술의 비약적인 발전은 인공지능을 단순한 알고리즘의 집합에서 '스스로 학습하는 존재'로 탈바꿈시켰다. 초기 인공지능이 바둑이나 체스 같은 정해진 규칙 안에서 승리하는 법을 배웠다면, 현재의 생성형 AI는 수조 개의 데이터를 학습하여 인간과 유사한 문장을 구사하고, 예술 작품을 창조하며, 복잡한 코드를 설계한다. 이러한 변화는 단순히 '똑똑한 도구'의 등장을 넘어, 정보의 생산과 소비 방식 자체를 뒤흔드는 패러다임의 전환을 의미한다. 이제 지식은 소유하는 것이 아니라, AI라는
자체를 뒤흔드는 패러다임의 전환을 의미한다. 이제 지식은 소유하는 것이 아니라, AI라는 인터페이스를 통해 어떻게 끌어내느냐(Prompt Engineering)의 문제가 되었다.
산업 구조의 재편과 새로운 기회 경제적 관점에서 인공지능은 생산성을 극대화하는 기폭제다. 제조, 물류, 금융, 의료 등 모든 산업 분야에서 AI는 효율성을 높이고 비용을 절감한다. 특히 의료 분야에서 AI는 방대한 임상 데이터를 분석해 인간 의사가 발견하지 못한 질병의 전조를 찾아내고, 신약 개발 기간을 획기적으로 단축하고 있다. 물론 이러한 효율성이 일자리 감소에 대한 공포를 불러일으키는 것도 사실이다. 단순 반복적인 업무는 AI와 로봇이 대체할 가능성이 크기 때문이다. 그러나 역사적으로 기술 혁신은 언제나 기존의 직업군을
대체할 가능성이 크기 때문이다. 그러나 역사적으로 기술 혁신은 언제나 기존의 직업군을 소멸시키는 동시에, 데이터 과학

#### 3) PyPDFLoader
- PyMuPDF가 속도가 더 빠르지만 상업용으로 활용시 문제가 될 수 있어서, 상업용 라이센스가 필요없는 PyPDFLoader가 접근성은 더 좋음

In [67]:
!pip install pypdf

In [69]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('data/기술보증기금과 한국경제.pdf')

# splitter 없이 사용하면 pdf 페이지별로 Document 객체로 반환됨
docs = loader.load_and_split()

print('페이지 수', len(docs))
print()
docs

페이지 수 5



[Document(metadata={'producer': 'Microsoft® Word Office 365용', 'creator': 'Microsoft® Word Office 365용', 'creationdate': '2019-06-27T22:46:56+09:00', 'author': 'HS', 'moddate': '2019-06-27T22:46:56+09:00', 'source': 'data/기술보증기금과 한국경제.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='페이지 1 / 5 \n \n기술보증기금과 한국경제 \n \nI. 기술보증기금 개요  \n1. 설립근거 : 기술보증기금법 \n \n- 설립목적(존재이유) \n✓ 기술보증기금을 설립하여 기술보증제도를 정착·발전시킴으로써 신 기술사업에 대한 자금의 \n공급을 원활하게 하고 나아가 국민 경제의 발전에 이바지함을 목적으로 함(기술보증기금법1\n조) \n✓ 설립 : 담보능력이 미약한 기업의 채무를 보증하게 하여 기업에 대한 자금 융통을 원활하게 하기 \n위하여 기술보증기금을 설립(법 12조) \n✓ 기금의 재원 : 정부 출연금의 예산은 중소벤처기업부 소관으로 함 \n \n☞ 기술보증기금은  \n✓ 기술력은 우수하지만 담보 부족한 중소기업의 \n✓ 기술성과 사업성 평가를 통해 기술보증을 지원하며, \n✓ 기술평가, 벤처이노비즈기업  인증, 중소기업 창업지원 등의 업무 수행 \n \n2. 주요개념1 \n \n업  무 내  용 \n기술보증 신기술사업자가 부담하는 금전 채무 보증.( 신용부족-담보부족 해결) \n \n금융회사 등으로부터 자금 대출을 받음으로써 금융회사 등에 대하여 부담하는 금전 채무를 \n기술보증기금이 기술보증서로 보증 \n 신기술사업자 - 기술을 개발하거나 이를 응용하여 사업화하는 중소기업(「중소기업기본법」에 \n따른 중소기업) 및 대통령령으로 정하는 기업 \n- "기업"이란 사업을 하는 개인 및 법인 \n신용보증 상시 사용하

In [70]:
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs_split_pdf = loader.load_and_split(text_splitter=pdf_splitter)
print('청크 수:', len(docs_split_pdf))
print()
docs_split_pdf

청크 수: 23



[Document(metadata={'producer': 'Microsoft® Word Office 365용', 'creator': 'Microsoft® Word Office 365용', 'creationdate': '2019-06-27T22:46:56+09:00', 'author': 'HS', 'moddate': '2019-06-27T22:46:56+09:00', 'source': 'data/기술보증기금과 한국경제.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='페이지 1 / 5 \n \n기술보증기금과 한국경제 \n \nI. 기술보증기금 개요  \n1. 설립근거 : 기술보증기금법 \n \n- 설립목적(존재이유) \n✓ 기술보증기금을 설립하여 기술보증제도를 정착·발전시킴으로써 신 기술사업에 대한 자금의 \n공급을 원활하게 하고 나아가 국민 경제의 발전에 이바지함을 목적으로 함(기술보증기금법1\n조) \n✓ 설립 : 담보능력이 미약한 기업의 채무를 보증하게 하여 기업에 대한 자금 융통을 원활하게 하기 \n위하여 기술보증기금을 설립(법 12조) \n✓ 기금의 재원 : 정부 출연금의 예산은 중소벤처기업부 소관으로 함'),
 Document(metadata={'producer': 'Microsoft® Word Office 365용', 'creator': 'Microsoft® Word Office 365용', 'creationdate': '2019-06-27T22:46:56+09:00', 'author': 'HS', 'moddate': '2019-06-27T22:46:56+09:00', 'source': 'data/기술보증기금과 한국경제.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='✓ 기금의 재원 : 정부 출연금의 예산은 중소벤처기업부 소관으로 함 \n \n☞ 기술보증기금은  \n✓ 기술력은 우수하지만 담보 부족한 중

In [ ]:
# <혼자> 네이버 블로그 내용 긁어오기. 그냥 주소로 하면 블로그 이름만 나옴. ID, 글번호를 형식에 맞게 넣어줘야 본문 추출 가능.
loader = WebBaseLoader('https://blog.naver.com/PostView.naver?blogId=bungto&logNo=224220125057')
blog = loader.load()

raw_txt = blog[0].page_content

# =========== 데이터 청소 해보자
# 1. 정규표현식
clean = ' '.join(re.findall(r'[\w]+', raw_txt))
clean

# 근데 공감 0, 칭찬 0 같은 애들 있음....

'오랜만에 설빙 티라미수 설빙 맛있어요 네이버 블로그 NAVER 블로그 백설의 유니크노트 Unique note 블로그 검색 이 블로그에서 검색 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 첫 댓글을 남겨보세요 공유하기 메뉴 바로가기 본문 바로가기 내 블로그 이웃블로그 블로그 홈 로그인 백설의 유니크노트 Unique note 블로그 메뉴 prologue blog 포토로그 비디오로그 map library tag guest blog 글쓰기 가벼운 글쓰기툴 퀵에디터가 오픈했어요 전체보기 2 024개의 글 전체보기목록열기 Food 오랜만에 설빙 티라미수 설빙 맛있어요 자유로운영혼 백설 19시간 전 URL 복사 이웃추가 본문 기타 기능 공유하기 신고하기 이웃님들 백설입니다 지인과 실컷 놀다 그냥 헤어지기섭섭해서 빙수 먹으러 설빙 의왕역점으로 고고 오늘은 색다르게 티라미수설빙과메론 인절미 반반 빙수 주문 메론빙수 메론이 적어서 좀 실망 그래도 맛은 있으니까 인절미빙수는 역시 맛있어 인철미 가루가 좀 약하네 반반이라 그런가 타라미수 설빙 티라미수 한덩이가 수저로 떠먹으니 시원하고 달콤하고 이것도 맛있네요 디저트로 딱인 빙수 한번 드셔보셔요 그럼 오늘도행복하세요 설빙 설빙의왕역점 티라미수설빙 인절미빙수 메론빙수 설빙 의왕역점 경기도 의왕시 부곡중앙남2길 1 2층 202호 태그 취소 확인 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 이 글에 공감한 블로거 열고 닫기 댓글 쓰기 이 글에 댓글 단 블로거 열고 닫기 인쇄 댓글쓰기 1 1 이전 다음 이 블로그 전체 카테고리 글 전체글 보기 글 목록 글 제목 작성일 화면 최상단으로 이동 title 오랜만에 설빙 티라미수 설빙 맛있어요 source https blog naver com bungto 224220125057 blogName 백설의 유 domainIdOrBlogId bungto

In [ ]:
!pip install html2text

In [ ]:
# 2. 제미나이가 버튼 같은거 자동으로 삭제해준대서 써봤는데.. 성능 구림...

from langchain_community.document_transformers import Html2TextTransformer

html2text = Html2TextTransformer()
blog_transformed = html2text.transform_documents(blog)

print(blog_transformed[0].page_content)

오랜만에 설빙~~~ 티라미수 설빙~~~ 맛있어요~^^ : 네이버 블로그 NAVER 블로그 백설의 유니크노트(Unique note) 블로그 검색 이 블로그에서 검색 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 첫 댓글을 남겨보세요 공유하기 메뉴 바로가기 본문 바로가기 내 블로그 이웃블로그 블로그 홈 로그인 백설의 유니크노트(Unique note) 블로그 메뉴 prologue blog 포토로그 비디오로그 map library tag guest blog 글쓰기 가벼운 글쓰기툴 퀵에디터가 오픈했어요! 전체보기 2,024개의 글 전체보기목록열기 Food 오랜만에 설빙~~~ 티라미수 설빙~~~ 맛있어요~^^ 자유로운영혼 백설 ・ 19시간 전 URL 복사 이웃추가 본문 기타 기능 공유하기 신고하기 ​이웃님들~!!!백설입니다~^^/​​ ​​지인과 실컷 놀다 그냥 헤어지기섭섭해서 빙수 먹으러 설빙 의왕역점으로~~고고~~^^​​​ ​​​오늘은 색다르게 티라미수설빙과메론 인절미 반반 빙수 주문~~~​​​ ​​​메론빙수~~~메론이 적어서 좀 실망~~그래도 맛은 있으니까~^^​​​ ​​​인절미빙수는 역시~~~맛있어~^^인철미 가루가 좀 약하네~~반반이라 그런가~^^;;​​​​ ​​​타라미수 설빙~~~티라미수 한덩이가~~~수저로 떠먹으니 ~~~~시원하고~~ 달콤하고~~~이것도 맛있네요~^^​​​ ​​​디저트로 딱인 빙수~~~한번 드셔보셔요~^^​​​그럼 오늘도행복하세요~!!!!​​ ​​#설빙 #설빙의왕역점 #티라미수설빙#인절미빙수 #메론빙수 ####​​​ 설빙 의왕역점 경기도 의왕시 부곡중앙남2길 1 2층 202호 ​​​​​ 태그 취소 확인 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 공감 0 공감 0 칭찬 0 감사 0 웃김 0 놀람 0 슬픔 0 이 글에 공감한 블로거 열고 닫기 댓글 쓰기 이 글에 댓글 단 블로거 열고 닫기 인쇄 댓글쓰기 1/1 이전 다음 이 블로그 전체 카테고리 글 전체글 보기

In [ ]:
# bs의 수프 스트레이너? 오 굿...

from langchain_community.document_loaders import WebBaseLoader
import bs4


loader = WebBaseLoader(
    web_path=('https://blog.naver.com/PostView.naver?blogId=bungto&logNo=224220125057'),
    # bs_kwargs : bs 세부사항 설정
    bs_kwargs={
        'parse_only': bs4.SoupStrainer(
            class_=["se-main-container", "se-module-text"]    # 네이버 본문 전용 클래스
        )
    }
)

blog = loader.load()

print(blog[0].page_content)






​이웃님들~!!!백설입니다~^^/​​




 









 



​​지인과 실컷 놀다 그냥 헤어지기섭섭해서 빙수 먹으러 설빙 의왕역점으로~~고고~~^^​​​




 















​​​오늘은 색다르게 티라미수설빙과메론 인절미 반반 빙수 주문~~~​​​




 



























​​​메론빙수~~~메론이 적어서 좀 실망~~그래도 맛은 있으니까~^^​​​




 















​​​인절미빙수는 역시~~~맛있어~^^인철미 가루가 좀 약하네~~반반이라 그런가~^^;;​​​​




 



























​​​타라미수 설빙~~~티라미수 한덩이가~~~수저로 떠먹으니 ~~~~시원하고~~ 달콤하고~~~이것도 맛있네요~^^​​​




 















​​​디저트로 딱인 빙수~~~한번 드셔보셔요~^^​​​그럼 오늘도행복하세요~!!!!​​




 









 



​​#설빙 #설빙의왕역점 #티라미수설빙#인절미빙수 #메론빙수 ####​​​




 







설빙 의왕역점
경기도 의왕시 부곡중앙남2길 1 2층 202호





 



​​​​​




 


In [ ]:
# 정규표현식으로 '진짜 단어'들만 뽑기. 근데 띄어쓰기가 이상..
words = re.findall(r'[\w]+', blog[0].page_content)
final_text = ' '.join(words)
final_text

'이웃님들 백설입니다 지인과 실컷 놀다 그냥 헤어지기섭섭해서 빙수 먹으러 설빙 의왕역점으로 고고 오늘은 색다르게 티라미수설빙과메론 인절미 반반 빙수 주문 메론빙수 메론이 적어서 좀 실망 그래도 맛은 있으니까 인절미빙수는 역시 맛있어 인철미 가루가 좀 약하네 반반이라 그런가 타라미수 설빙 티라미수 한덩이가 수저로 떠먹으니 시원하고 달콤하고 이것도 맛있네요 디저트로 딱인 빙수 한번 드셔보셔요 그럼 오늘도행복하세요 설빙 설빙의왕역점 티라미수설빙 인절미빙수 메론빙수 설빙 의왕역점 경기도 의왕시 부곡중앙남2길 1 2층 202호'